# Lección 11 - Protocolo Agente a Agente (A2A)


## Configuración


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ¿Qué es el Protocolo A2A?

El **protocolo Agent-to-Agent (A2A)** es un estándar abierto que permite que los agentes de IA se comuniquen,
se descubran entre sí y colaboren, incluso cuando están construidos sobre diferentes frameworks o alojados
por distintos servicios.

Conceptos clave:

- **Descubrimiento** – Los agentes publican una *Tarjeta de Agente* que describe sus capacidades, facilitando que otros agentes (o los orquestadores) encuentren al especialista adecuado para una tarea.
- **Intercambio de Mensajes** – Los agentes intercambian mensajes estructurados a través de un protocolo común, de modo que una solicitud de un agente pueda ser entendida y cumplida por otro sin importar su implementación interna.
- **Ciclo de Vida de la Tarea** – A2A define estados como *enviado*, *en progreso*, *completado* y *fallido*, ofreciendo al orquestador una visibilidad completa sobre el avance de una tarea delegada.

En esta lección simulamos la colaboración al estilo A2A conectando tres agentes de viajes especializados
en un flujo de trabajo donde cada agente aporta su experiencia y pasa los resultados al siguiente.


## Creando Agentes de Viajes Especializados


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Colaboración Multiagente a través del Flujo de Trabajo

Conectamos los tres agentes en un flujo de trabajo secuencial que refleja el paso de mensajes A2A:

1. **CurrencyExchangeAgent** recibe la solicitud del usuario y produce orientación sobre divisas.
2. **ActivityPlannerAgent** recibe el contexto enriquecido y añade recomendaciones de actividades.
3. **TravelManagerAgent** sintetiza ambas entradas en un informe final de viaje.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Comprendiendo A2A en Producción

En un entorno de producción, el protocolo A2A desbloquea potentes escenarios entre servicios:

| Capacidad | Descripción |
|---|---|
| **Interoperabilidad entre frameworks** | Un agente construido con un framework puede delegar tareas a un agente construido con cualquier otro framework compatible con A2A, permitiendo una verdadera interoperabilidad entre organizaciones. |
| **Límites del servicio** | Los agentes pueden residir en microservicios separados, regiones de la nube, o incluso en organizaciones diferentes, mientras colaboran sin problemas. |
| **Descubrimiento dinámico** | Un orquestador puede consultar un registro de Agent Cards en tiempo real para encontrar al especialista más adecuado para una subtarea determinada. |
| **Transmisión y notificaciones push** | A2A soporta Eventos Enviados por el Servidor (SSE) para actualizaciones de progreso en tiempo real y notificaciones push para tareas de larga duración. |

El flujo de trabajo que construimos arriba es una versión simplificada, en proceso, de este patrón. En un despliegue real, cada agente expondría un endpoint HTTP, publicaría una Agent Card y se comunicaría mediante el protocolo JSON-RPC de A2A.


## Resumen

En esta lección aprendiste:

1. **Qué es el protocolo A2A** — un estándar abierto para el descubrimiento, mensajería y gestión de tareas entre agentes.
2. **Cómo crear agentes especializados** — un agente de Cambio de Divisas, un agente Planificador de Actividades y un orquestador Gestor de Viajes.
3. **Cómo conectar agentes en un flujo de trabajo** — usando `WorkflowBuilder` para modelar el paso secuencial de mensajes entre agentes.
4. **Cómo funciona A2A en producción** — permitiendo la colaboración entre diferentes frameworks y servicios con descubrimiento dinámico y actualizaciones en streaming.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso legal**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automáticas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional realizada por humanos. No nos responsabilizamos por malentendidos o interpretaciones erróneas derivados del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
